# Build private Kaggle add-on — STB1 seed-42 teacher
Jalankan **Runtime → Run all** di Colab. Notebook ini hanya menyalin dan memvalidasi frozen `STB1_seed42/weights/best.pt` dari proyek Drive ke add-on Kaggle kecil. Tidak melatih dan tidak membuka locked test.

Output private dataset: `faruq-v3-stb1-teacher-addon-v1`.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)
import importlib,json,os,shutil,subprocess,sys,time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/stb-guided-robust-wav-yolo'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','kaggle'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('REPO COMMIT:',COMMIT)
print('ULTRALYTICS:',__import__('ultralytics').__version__)
from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.experiments.prepare_stb_guided_kaggle import build_stb_guided_teacher_addon
PROJECT_ROOT=resolve_drive_project_root(); BUNDLE=Path('/content/stb1-teacher-addon-v1')
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
manifest=build_stb_guided_teacher_addon(PROJECT_ROOT,BUNDLE)
assert manifest['test_images_included'] is False
print('PROJECT:',PROJECT_ROOT)
print('BUNDLE:',BUNDLE)
print(json.dumps(manifest,indent=2))


In [ ]:
username=userdata.get('KAGGLE_USERNAME'); token=userdata.get('KAGGLE_API_TOKEN')
assert username and token, 'Aktifkan secret KAGGLE_USERNAME dan KAGGLE_API_TOKEN.'
os.environ['KAGGLE_USERNAME']=username; os.environ['KAGGLE_API_TOKEN']=token; os.environ['KAGGLE_KEY']=token
dataset_id=f'{username}/faruq-v3-stb1-teacher-addon-v1'
metadata={'title':'Faruq V3 STB1 Teacher Addon V1','id':dataset_id,'licenses':[{'name':'other'}],'isPrivate':True}
(BUNDLE/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
message='Frozen STB1 seed42 teacher addon; validation-development artifact only; no test data'
version=subprocess.run(['kaggle','datasets','version','-p',str(BUNDLE),'-m',message],text=True,capture_output=True)
if version.returncode!=0:
    combined=(version.stdout+'\n'+version.stderr).lower(); missing=any(t in combined for t in ('not found','404','does not exist'))
    if not missing:
        print(version.stdout); print(version.stderr); raise RuntimeError(f'Kaggle dataset version gagal: {version.returncode}')
    subprocess.run(['kaggle','datasets','create','-p',str(BUNDLE)],check=True)
else:
    print(version.stdout)
print('UPLOAD SELESAI:',f'https://www.kaggle.com/datasets/{dataset_id}')
print('Attach dua input pada notebook S2 Kaggle: faruq-v3-experiment-core-v1 + faruq-v3-stb1-teacher-addon-v1.')
